In [2]:
# @title Github Data Loader, Tokenizer Training, and Dataset Builder

import os

try:
    import google.colab
    REPO_URL = "https://github.com/wtheisen/nd-cse-10124-lectures.git"

    REPO_NAME = "/content/nd-cse-10124-lectures"
    L_PATH = "nd-cse-10124-lectures"

    %cd /content/
    !rm -r {REPO_NAME}

    # Clone repo
    if not os.path.exists(REPO_NAME):
        !git clone {REPO_URL}

        # cd into the data folder
        %cd {L_PATH}
        !pwd

        !pip install torch transformers regex tqdm

        !python scripts/import_gpt2_small.py \
  --out Datasets/gpt2_small_converted.pt \
  --tokenizer_dir Datasets/gpt2_tokenizer_assets

except ImportError:
    print("Unable to download repo, either:")
    print("\tA.) You're not on colab")
    print("\tB.) It has already been cloned")

!pwd

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

/content
Cloning into 'nd-cse-10124-lectures'...
remote: Enumerating objects: 445, done.
remote: Counting objects: 100% (34/34), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 445 (delta 8), reused 23 (delta 7), pack-reused 411 (from 1)
Receiving objects: 100% (445/445), 57.03 MiB | 12.80 MiB/s, done.
Resolving deltas: 100% (271/271), done.
/content/nd-cse-10124-lectures
/content/nd-cse-10124-lectures
config.json: 100% 665/665 [00:00<00:00, 2.66MB/s]
model.safetensors: 100% 548M/548M [00:03<00:00, 140MB/s]
Loading weights: 100% 148/148 [00:00<00:00, 1667.33it/s, Materializing param=transformer.wte.weight]
GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
generation_config.json: 100% 124/124 [00:00<00:00, 507kB/s]
tokenizer_config.json: 10

In [4]:
import torch
from irishGPT.irishChat import IrishChat
from irishGPT.gpt2_tokenizer import GPT2Tokenizer

from transformers import GPT2Tokenizer

tok_hf = GPT2Tokenizer.from_pretrained("gpt2")
tok_hf.save_pretrained("Datasets/gpt2_tokenizer_assets")

# Paths
ckpt_path = "Datasets/gpt2_small_converted.pt"
vocab_path = "Datasets/gpt2_tokenizer_assets/vocab.json"
merges_path = "Datasets/gpt2_tokenizer_assets/merges.txt"

# Load tokenizer + model
tok = GPT2Tokenizer(vocab_path, merges_path)
model = IrishChat.gpt2_small()
model.load_converted_gpt2_checkpoint(ckpt_path)

# Prompt -> tokens
prompt = "Explain what attention does in one paragraph."
prompt_ids = tok.encode(prompt)

# Generate
out_ids = model.chat(
    prompt_ids,
    max_new_tokens=120,
    temperature=0.8,
    eos_token_id=50256,  # GPT-2 endoftext
)

# Decode
print(tok.decode(out_ids))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Exception: Error while initializing BPE: No such file or directory (os error 2)

In [ ]:
# Mini ChatGPT-style UI for Colab (ipywidgets) — GPT2-wired
# Assumes you already ran:
#   from irishGPT.irishChat import IrishChat
#   from irishGPT.gpt2_tokenizer import GPT2Tokenizer
#   chat = IrishChat.gpt2_small()
#   chat.load_converted_gpt2_checkpoint("Datasets/gpt2_small_converted.pt")
#   tok = GPT2Tokenizer("Datasets/gpt2_tokenizer_assets/vocab.json",
#                       "Datasets/gpt2_tokenizer_assets/merges.txt")

!pip -q install ipywidgets

from google.colab import output
output.enable_custom_widget_manager()

import ipywidgets as widgets
import html

# --------- STATE ----------
history = []  # list of (role, text), role in {"user","assistant"}

# --------- WIDGET STYLES ----------
style = widgets.HTML(value="""
<style>
.chat-wrap { font-family: system-ui, -apple-system, Segoe UI, Roboto, sans-serif; }
.chat-log  { height: 420px; overflow-y: auto; border: 1px solid #ddd; border-radius: 14px; padding: 12px; background: #fafafa; }
.msg { display: flex; margin: 10px 0; }
.bubble { max-width: 85%; padding: 10px 12px; border-radius: 14px; line-height: 1.35; white-space: pre-wrap; }
.user { justify-content: flex-end; }
.user .bubble { background: #dbeafe; border: 1px solid #bfdbfe; }
.assistant { justify-content: flex-start; }
.assistant .bubble { background: #ffffff; border: 1px solid #e5e7eb; }
.meta { font-size: 12px; color: #6b7280; margin-top: 6px; }
</style>
""")

# --------- UI ELEMENTS ----------
log = widgets.HTML(value="")
prompt = widgets.Text(
    placeholder="Message IrishGPT…",
    layout=widgets.Layout(width="60%")
)
send = widgets.Button(description="Send", button_style="primary")
clear_btn = widgets.Button(description="Clear")

temp = widgets.FloatSlider(
    value=0.8, min=0.1, max=1.5, step=0.05,
    description="Temp", continuous_update=False,
    layout=widgets.Layout(width="300px")
)

max_new = widgets.IntSlider(
    value=200, min=16, max=512, step=16,
    description="MaxNew", continuous_update=False,
    layout=widgets.Layout(width="320px")
)

status = widgets.HTML(value="<div class='meta'>Ready.</div>")

# --------- RENDERING ----------
def render_history():
    parts = ["<div class='chat-wrap'><div class='chat-log'>"]
    for role, text in history:
        safe = html.escape(text)
        cls = "user" if role == "user" else "assistant"
        parts.append(f"<div class='msg {cls}'><div class='bubble'>{safe}</div></div>")
    parts.append("</div></div>")
    log.value = "".join(parts)

def add_message(role, text):
    history.append((role, text))
    render_history()

# --------- GENERATION (GPT-2 wiring) ----------
def generate_reply(user_text: str) -> str:
    prompt_tokens = tok.encode(user_text)
    out_tokens = chat.chat(
        prompt_tokens,
        max_new_tokens=int(max_new.value),
        temperature=float(temp.value),
        eos_token_id=50256,  # GPT-2 endoftext
    )

    # show only newly generated continuation
    gen_tokens = out_tokens[len(prompt_tokens):]
    return tok.decode(gen_tokens)

# --------- HANDLERS ----------
def on_send(_=None):
    user_text = prompt.value.strip()
    if not user_text:
        return
    prompt.value = ""

    add_message("user", user_text)
    status.value = "<div class='meta'>Generating…</div>"

    try:
        reply = generate_reply(user_text)
        add_message("assistant", reply if reply.strip() else "[empty response]")
        status.value = "<div class='meta'>Ready.</div>"
    except Exception as e:
        add_message("assistant", f"[error] {type(e).__name__}: {e}")
        status.value = "<div class='meta'>Error.</div>"

def on_clear(_=None):
    history.clear()
    render_history()
    status.value = "<div class='meta'>Cleared.</div>"

send.on_click(on_send)
clear_btn.on_click(on_clear)
prompt.on_submit(on_send)

controls = widgets.HBox([send, clear_btn, temp, max_new], layout=widgets.Layout(width="100%"))
ui = widgets.VBox(
    [style, log, prompt, controls, status],
    layout=widgets.Layout(width="40%", margin="0 auto")
)

render_history()
display(ui)